In [ ]:
import logging
import logging.config
import os
from pathlib import Path

import httpx
import pandas as pd
from unidecode import unidecode

from graal.summary.blind_eval_project import CHOOSE_BEST_OBJECT_COLUMN, BlindEvalProject
from graal.summary.llm_clients import AlbertAPIClient, ChatGPTAPIClient
from graal.utils.amendment_pre_processor import AmendmentPreProcessor
from graal.utils.text_utils import remove_sentences_starting_with

logging.config.fileConfig("logging.conf")

# MAIN


DATA_FOLDER = os.getenv("DATA_FOLDER", "data")
METRICS = ["Correct", "Complet", "Concis"]

COLUMN_ORDER = ["ID", "Exposé amdt", "Corps amdt"]
for i in range(1, 3):
    COLUMN_ORDER.append(f"Objet {i}")
    for metric in METRICS:
        COLUMN_ORDER.append(f"{i} - {metric}")
COLUMN_ORDER.append(f"{CHOOSE_BEST_OBJECT_COLUMN}")

EXCEL_OUTPUT_FILE = Path(f"{DATA_FOLDER}/blind_eval_summaries/eval_aveugle_objets.xlsx")
PROJECT_SAVE_LOCATION = Path(
    f"{DATA_FOLDER}/blind_eval_summaries/project_eval_aveugle_objets.pkl"
)
GRAAL_CONFIG_FILE = Path(
    f"{DATA_FOLDER}/config_graal/Fichier de configuration GRAAL - latest.xlsx"
)
FILE_TO_SAMPLE_FROM = Path(
    f"{DATA_FOLDER}/exports_lectures/PLFSS 2025/BDD_AN_L1_SP_Amendements_copie_valeurs.xlsx"
)

attribution_mappings_excel = pd.read_excel(GRAAL_CONFIG_FILE, sheet_name=None)

try:
    project = BlindEvalProject.load_from_disk(PROJECT_SAVE_LOCATION)
    logging.info(f"Project '{PROJECT_SAVE_LOCATION}' successfully loaded")
except (FileNotFoundError, EOFError):
    logging.info(f"Creating new project '{PROJECT_SAVE_LOCATION}'")
    amendments_df = AmendmentPreProcessor.load_amendments_excel(
        input_files=[FILE_TO_SAMPLE_FROM]
    )
    amendments_df["Corps amdt"] = amendments_df["Corps amdt"].apply(
        lambda text: remove_sentences_starting_with(
            text,
            patterns=[
                unidecode("la perte de recettes"),
                unidecode(
                    "la charge pour l état et les collectivités territoriales est compensée"
                ),
            ],
        )
    )
    amendments_df = AmendmentPreProcessor.remap_columns_in_json_amendments(
        amendments_df
    )

    amendments_df = amendments_df[amendments_df["Objet amdt"].str.strip() != ""]
    amendments_df = amendments_df[
        ~amendments_df["Objet amdt"].str.contains(
            "Amendement rédactionnel|Supprimer cet article|Supprimer l'article|Amendement de coordination|Modifier l'alinea|Modifier la rédaction|irr\?",
            na=False,
        )
    ]
    amendments_df = amendments_df[
        ~amendments_df["Exposé amdt"].str.contains(
            "amendement de correction d'une erreur matérielle",
            na=False,
        )
    ]

    project = BlindEvalProject(
        amendments_df=amendments_df,
        metrics=METRICS,
        config_prompt=attribution_mappings_excel["Prompt Objet"].to_string(),
        rate_limiting_config={"albert": 10},
    )

llama_70B_client = AlbertAPIClient(  # noqa: N816
    base_url=httpx.URL(os.environ["ETALAB_BASE_URL"]),
    api_key=os.environ["ETALAB_API_KEY"],
    model_name=os.environ["ETALAB_MODEL_NAME"],
)

OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
gpt_4_turbo_client = ChatGPTAPIClient(model_name="gpt-4-turbo", api_key=OPENAI_API_KEY)

gpt_4o_mini_client = ChatGPTAPIClient(model_name="gpt-4o-mini", api_key=OPENAI_API_KEY)

llm_clients = {
    "llama-3.1-70b-instruct": llama_70B_client,
    "gpt-4-turbo": gpt_4_turbo_client,
    "gpt-4o-mini": gpt_4o_mini_client,
}

project.add_next_n_rows(80, llm_clients)

project.to_excel(
    output_file=EXCEL_OUTPUT_FILE, column_order=COLUMN_ORDER, excluded_ids=[]
)
project.dump_to_disk(output_file=PROJECT_SAVE_LOCATION)
# Open the Excel file
os.system(f'open "{EXCEL_OUTPUT_FILE}"')  # noqa: S605
project.mapping_obj_to_author

INFO - Creating new project 'data/blind_eval_summaries/project_eval_aveugle_objets.pkl'
INFO - albert_qlbcg is generating a summary
INFO - chatgpt_eezza is generating a summary
INFO - chatgpt_gdkjw is generating a summary
INFO - albert_qlbcg is generating a summary
INFO - chatgpt_eezza is generating a summary
INFO - chatgpt_gdkjw is generating a summary
INFO - albert_qlbcg is generating a summary
INFO - chatgpt_eezza is generating a summary
INFO - chatgpt_gdkjw is generating a summary
INFO - albert_qlbcg is generating a summary
INFO - chatgpt_eezza is generating a summary
INFO - chatgpt_gdkjw is generating a summary
INFO - albert_qlbcg is generating a summary
INFO - chatgpt_eezza is generating a summary
INFO - chatgpt_gdkjw is generating a summary
INFO - albert_qlbcg is generating a summary
INFO - chatgpt_eezza is generating a summary
INFO - chatgpt_gdkjw is generating a summary
INFO - albert_qlbcg is generating a summary
INFO - chatgpt_eezza is generating a summary
INFO - chatgpt_gdkj

{0: {'Objet 1': 'Expert', 'Objet 2': 'llama-3.1-70b-instruct'},
 1: {'Objet 1': 'gpt-4-turbo', 'Objet 2': 'Expert'},
 2: {'Objet 1': 'Expert', 'Objet 2': 'gpt-4o-mini'},
 3: {'Objet 1': 'llama-3.1-70b-instruct', 'Objet 2': 'Expert'},
 4: {'Objet 1': 'Expert', 'Objet 2': 'gpt-4-turbo'},
 5: {'Objet 1': 'Expert', 'Objet 2': 'gpt-4o-mini'},
 6: {'Objet 1': 'llama-3.1-70b-instruct', 'Objet 2': 'Expert'},
 7: {'Objet 1': 'Expert', 'Objet 2': 'gpt-4-turbo'},
 8: {'Objet 1': 'gpt-4o-mini', 'Objet 2': 'Expert'},
 9: {'Objet 1': 'Expert', 'Objet 2': 'llama-3.1-70b-instruct'},
 10: {'Objet 1': 'Expert', 'Objet 2': 'gpt-4-turbo'},
 11: {'Objet 1': 'Expert', 'Objet 2': 'gpt-4o-mini'},
 12: {'Objet 1': 'Expert', 'Objet 2': 'llama-3.1-70b-instruct'},
 13: {'Objet 1': 'gpt-4-turbo', 'Objet 2': 'Expert'},
 14: {'Objet 1': 'Expert', 'Objet 2': 'gpt-4o-mini'},
 15: {'Objet 1': 'llama-3.1-70b-instruct', 'Objet 2': 'Expert'},
 16: {'Objet 1': 'gpt-4-turbo', 'Objet 2': 'Expert'},
 17: {'Objet 1': 'Expert',